# Train the crop health models (Colab GPU)

Trains one model per crop (cassava, maize, beans) with the configs in `crop_health_model/configs/`, tests each one, and shows the per-class results and confusion matrices. All training logic is in the codebase; this notebook only sets up Colab and runs the same commands you would run locally.

**Before you start**
- Runtime > Change runtime type > **T4 GPU**.
- The dataset zip must already be in Drive (made by `prepare_data_colab.ipynb`).
- Run with `SMOKE_TEST = True` first (1,000 images, 1 epoch, a few minutes) to check everything works, then set it to `False` for the real training.
- Runs are saved to Drive (`crop-health/runs/`), so checkpoints and results survive a disconnect. If the session drops, set `CROPS` to the crops not finished yet and run again.
- Free Colab limits GPU time. If you hit the limit, train one crop per session.

## 1. Settings

In [ ]:
CROPS = ["cassava", "maize", "beans"]  # crops to train in this session
SMOKE_TEST = True  # True: quick check on 1,000 images for 1 epoch. False: full training.

DRIVE_DIR = "/content/drive/MyDrive/crop-health"
DRIVE_ZIP = f"{DRIVE_DIR}/crop-data.zip"
# Smoke test runs are kept apart so they don't mix with real results
RUNS_DIR = f"{DRIVE_DIR}/runs-smoke" if SMOKE_TEST else f"{DRIVE_DIR}/runs"

REPO_URL = "https://github.com/Gabin-ishimwe/crop-health-model.git"
BRANCH = "feat/per-crop-training"  # change to "main" once the PR is merged
REPO_DIR = "/content/crop-health-model"

## 2. Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run again."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 3. Mount Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
assert os.path.exists(DRIVE_ZIP), f"No dataset zip at {DRIVE_ZIP}. Run prepare_data_colab.ipynb first."
print(f"Dataset zip found ({os.path.getsize(DRIVE_ZIP) / 1024**3:.1f} GB)")

## 4. Get the code and install it

Colab already has a GPU build of torch, torchvision, pandas, matplotlib and tensorboard. Only the missing packages are installed, and the project is installed with `--no-deps` so Colab's torch is not replaced by the older version pinned in `pyproject.toml`.

In [ ]:
if not os.path.exists(REPO_DIR):
    !GIT_TERMINAL_PROMPT=0 git clone -q --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull -q
!git log --oneline -1

!pip install -q lightning torchmetrics "jsonargparse[signatures]" rarfile
!pip install -q --no-deps -e .

## 5. Load the dataset

Copies the zip from Drive to Colab's disk and extracts it into `.data` (a few minutes; skipped if already done in this session).

In [ ]:
!python -m crop_health_model.data.make_dataset --drive-zip "{DRIVE_ZIP}"

import pandas as pd

df = pd.read_csv(".data/annotations.csv")
print(f"\n{len(df)} images")
print(df.groupby(["crop_type", "label"]).size().rename("images").to_string())

## 6. Save runs to Drive

Training writes to `tb_logs/`. Pointing it at Drive keeps checkpoints and results if Colab disconnects.

In [ ]:
os.makedirs(RUNS_DIR, exist_ok=True)
if os.path.islink("tb_logs"):
    os.unlink("tb_logs")
elif os.path.exists("tb_logs"):
    raise RuntimeError("tb_logs is a real folder here; move it before linking to Drive")
os.symlink(RUNS_DIR, "tb_logs")
print(f"tb_logs -> {RUNS_DIR}")

## 7. Train

Trains each crop in `CROPS` one after the other. One data loading worker is used per CPU core (2 on free Colab, more on Pro High-RAM). Mixed precision (`16-mixed`) speeds up training on the GPU.

Rough times on a free T4 with full training: cassava ~30 min, maize ~1 h, beans ~20 min (training stops early when the validation loss stops improving).

In [ ]:
import glob


def latest_version(crop):
    """The most recent run folder for a crop, e.g. tb_logs/maize/version_2."""
    versions = glob.glob(f"tb_logs/{crop}/version_*")
    return max(versions, key=lambda p: int(p.rsplit("_", 1)[1])) if versions else None


# Image loading is the bottleneck, so use one loader per CPU core (2 on free Colab, more on Pro High-RAM)
WORKERS = os.cpu_count()
print(f"Using {WORKERS} data loader workers")
extra = "--data.limit 1000 --trainer.max_epochs 1" if SMOKE_TEST else ""
trained = {}
for crop in CROPS:
    print(f"\n===== Training {crop} =====")
    !python crop_health_model/scripts/train.py --config crop_health_model/configs/config_{crop}.yaml fit --data.num_workers {WORKERS} --trainer.precision 16-mixed {extra}
    if _exit_code != 0:
        print(f"Training {crop} failed, see the error above")
        continue
    trained[crop] = latest_version(crop)
    print(f"{crop} run saved in {trained[crop]}")

## 8. Test

Evaluates each trained model on its test set. The results go into a new run folder next to the training run.

In [ ]:
tested = {}
for crop, run_dir in trained.items():
    ckpt = glob.glob(f"{run_dir}/checkpoints/*.ckpt")[0]
    print(f"\n===== Testing {crop} ({ckpt}) =====")
    !python crop_health_model/scripts/train.py test --config {run_dir}/config.yaml --ckpt_path "{ckpt}" --data.num_workers {WORKERS}
    if _exit_code == 0:
        tested[crop] = latest_version(crop)

## 9. Results

In [ ]:
from IPython.display import Image, display

for crop, test_dir in tested.items():
    print(f"\n===== {crop} (training run: {trained[crop]}) =====")
    report = pd.read_csv(f"{test_dir}/test_per_class.csv")
    display(report.style.format({"recall": "{:.3f}", "precision": "{:.3f}"}).hide(axis="index"))
    display(Image(f"{test_dir}/test_confusion_matrix.png", width=450))

**Reading the results**
- **recall**: share of that class's test photos the model recognised. A low value means the disease is often missed.
- **precision**: when the model says this class, how often it is right.
- In the confusion matrix, rows are the true class and columns the predicted one. Numbers off the diagonal are mistakes.

## 10. Training curves (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tb_logs

## Where everything is

In Drive, `crop-health/runs/<crop>/version_N/` holds for each training run:
- `checkpoints/best_model.pt`: the trained weights (for the API)
- `index_to_name.json`: class names in output order
- `model_handler.py`, `model_script.py`: files for the TorchServe model archive
- `config.yaml`: the exact settings used

The test run folder next to it holds `test_per_class.csv`, `test_confusion_matrix.csv` and `test_confusion_matrix.png`.